In [53]:
from pathlib import Path
import numpy as np
import torch
from typing import List
from torch.nn.utils.rnn import pad_sequence
from mltrainer import rnn_models, Trainer
from torch import optim

from mads_datasets import datatools
import mltrainer
mltrainer.__version__

'0.2.7'

# 1 Iterators
We will be using an interesting dataset. [link](https://tev.fbk.eu/resources/smartwatch)

From the site:
> The SmartWatch Gestures Dataset has been collected to evaluate several gesture recognition algorithms for interacting with mobile applications using arm gestures. Eight different users performed twenty repetitions of twenty different gestures, for a total of 3200 sequences. Each sequence contains acceleration data from the 3-axis accelerometer of a first generation Sony SmartWatch™, as well as timestamps from the different clock sources available on an Android device. The smartwatch was worn on the user's right wrist. 


In [54]:
from mads_datasets import DatasetFactoryProvider, DatasetType
from mltrainer.preprocessors import PaddedPreprocessor
preprocessor = PaddedPreprocessor()

gesturesdatasetfactory = DatasetFactoryProvider.create_factory(DatasetType.GESTURES)
streamers = gesturesdatasetfactory.create_datastreamer(batchsize=32, preprocessor=preprocessor)
train = streamers["train"]
valid = streamers["valid"]

2026-04-08 13:44:57.166 | INFO     | mads_datasets.base:download_data:121 - Folder already exists at C:\Users\D110303\.cache\mads_datasets\gestures
100%|██████████| 651/651 [00:08<00:00, 72.75it/s]


In [55]:
len(train), len(valid)

(81, 20)

In [56]:
trainstreamer = train.stream()
validstreamer = valid.stream()
x, y = next(iter(trainstreamer))
x.shape, y

(torch.Size([32, 32, 3]),
 tensor([ 4,  8, 11, 13,  1,  1, 16,  6, 18, 14,  3, 11, 13, 16,  9, 13, 13,  6,
          3,  9,  2,  4,  8,  5, 17, 16,  2, 17,  9,  8,  2, 12]))

Can you make sense of the shape?
What does it mean that the shapes are sometimes (32, 27, 3), but a second time might look like (32, 30, 3)? In other words, the second (or first, if you insist on starting at 0) dimension changes. Why is that? How does the model handle this? Do you think this is already padded, or still has to be padded?


# 2 Excercises
Lets test a basemodel, and try to improve upon that.

Fill the gestures.gin file with relevant settings for `input_size`, `hidden_size`, `num_layers` and `horizon` (which, in our case, will be the number of classes...)

As a rule of thumbs: start lower than you expect to need!

In [57]:
from mltrainer import TrainerSettings, ReportTypes
from mltrainer.metrics import Accuracy

accuracy = Accuracy()


In [58]:
model = rnn_models.BaseRNN(
    input_size=3,
    hidden_size=64,
    num_layers=1,
    horizon=20,
)

Test the model. What is the output shape you need? Remember, we are doing classification!

In [59]:
yhat = model(x)
yhat.shape

torch.Size([32, 20])

Test the accuracy

In [60]:
accuracy(y, yhat)

0.0

What do you think of the accuracy? What would you expect from blind guessing?

Check shape of `y` and `yhat`

In [61]:
yhat.shape, y.shape

(torch.Size([32, 20]), torch.Size([32]))

And look at the output of yhat

In [62]:
yhat[0]

tensor([ 0.1774,  0.0366, -0.0145,  0.0935,  0.1303, -0.0439, -0.0113, -0.0681,
         0.0905,  0.1111,  0.0637,  0.1051, -0.0229,  0.0245, -0.0932, -0.0296,
         0.1019,  0.0219,  0.1160, -0.0206], grad_fn=<SelectBackward0>)

Does this make sense to you? If you are unclear, go back to the classification problem with the MNIST, where we had 10 classes.

We have a classification problem, so we need Cross Entropy Loss.
Remember, [this has a softmax built in](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) 

In [63]:
loss_fn = torch.nn.CrossEntropyLoss()
loss = loss_fn(yhat, y)
loss

tensor(2.9978, grad_fn=<NllLossBackward0>)

In [64]:
import torch
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
elif torch.cuda.is_available():
    device = "cuda:0"
    print("using cuda")
else:
    device = "cpu"
    print("using cpu")

# on my mac, at least for the BaseRNN model, mps does not speed up training
# probably because the overhead of copying the data to the GPU is too high
# so i override the device to cpu
device = "cpu"
# however, it might speed up training for larger models, with more parameters

using cpu


Set up the settings for the trainer and the different types of logging you want

In [65]:
settings = TrainerSettings(
    epochs=3, # increase this to about 100 for training
    metrics=[accuracy],
    logdir=Path("gestures"),
    train_steps=len(train),
    valid_steps=len(valid),
    reporttypes=[ReportTypes.TOML, ReportTypes.TENSORBOARD, ReportTypes.MLFLOW],
    scheduler_kwargs={"factor": 0.5, "patience": 5},
    earlystop_kwargs = {
        "save": False, # save every best model, and restore the best one
        "verbose": True,
        "patience": 5, # number of epochs with no improvement after which training will be stopped
        "delta": 0.0, # minimum change to be considered an improvement
    }
)
settings

epochs: 3
metrics: [Accuracy]
logdir: gestures
train_steps: 81
valid_steps: 20
reporttypes: [<ReportTypes.TOML: 'TOML'>, <ReportTypes.TENSORBOARD: 'TENSORBOARD'>, <ReportTypes.MLFLOW: 'MLFLOW'>]
optimizer_kwargs: {'lr': 0.001, 'weight_decay': 1e-05}
scheduler_kwargs: {'factor': 0.5, 'patience': 5}
earlystop_kwargs: {'save': False, 'verbose': True, 'patience': 5, 'delta': 0.0}

In [66]:
import torch.nn as nn
import torch
from torch import Tensor
from dataclasses import dataclass

@dataclass
class ModelConfig:
    input_size: int
    hidden_size: int
    num_layers: int
    output_size: int
    dropout: float = 0.0

class GRUmodel(nn.Module):
    def __init__(
        self,
        config,
    ) -> None:
        super().__init__()
        self.config = config
        self.rnn = nn.GRU(
            input_size=config.input_size,
            hidden_size=config.hidden_size,
            dropout=config.dropout,
            batch_first=True,
            num_layers=config.num_layers,
        )
        self.linear = nn.Linear(config.hidden_size, config.output_size)

    def forward(self, x: Tensor) -> Tensor:
        x, _ = self.rnn(x)
        last_step = x[:, -1, :]
        yhat = self.linear(last_step)
        return yhat

In [67]:
config = ModelConfig(
    input_size=3,
    hidden_size=64,
    num_layers=1,
    output_size=20,
    dropout=0.0,
)


In [74]:
import mlflow
from datetime import datetime

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("gestures")
modeldir = Path("gestures").resolve()
if not modeldir.exists():
    modeldir.mkdir(parents=True)

with mlflow.start_run():
    mlflow.set_tag("model", "modelname-here")
    mlflow.set_tag("dev", "your-name-here")
    config = ModelConfig(
        input_size=3,
        hidden_size=256,
        num_layers=2,
        output_size=20,
        dropout=0.12,
    )

    model = GRUmodel(
        config=config,
    )

    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=loss_fn,
        optimizer=optim.Adam,
        traindataloader=trainstreamer,
        validdataloader=validstreamer,
        scheduler=optim.lr_scheduler.ReduceLROnPlateau,
        device=device,
    )
    trainer.loop()

    if not settings.earlystop_kwargs["save"]:
        tag = datetime.now().strftime("%Y%m%d-%H%M-")
        modelpath = modeldir / (tag + "model.pt")
        torch.save(model, modelpath)

2026-04-08 13:51:06.744 | INFO     | mltrainer.trainer:dir_add_timestamp:24 - Logging to gestures\20260408-135106
2026-04-08 13:51:06.746 | INFO     | mltrainer.trainer:__init__:66 - Found earlystop_kwargs in settings.Set to None if you dont want earlystopping.
100%|██████████| 81/81 [00:03<00:00, 20.88it/s]
2026-04-08 13:51:10.924 | INFO     | mltrainer.trainer:report:215 - Epoch 0 train 2.4364 test 2.0265 metric ['0.2734']
100%|██████████| 81/81 [00:03<00:00, 21.57it/s]
2026-04-08 13:51:14.979 | INFO     | mltrainer.trainer:report:215 - Epoch 1 train 1.6763 test 1.2644 metric ['0.5297']
100%|██████████| 81/81 [00:03<00:00, 21.67it/s]
2026-04-08 13:51:19.074 | INFO     | mltrainer.trainer:report:215 - Epoch 2 train 0.9219 test 0.6289 metric ['0.8187']
100%|██████████| 3/3 [00:12<00:00,  4.11s/it]


Try to update the code above by changing the hyperparameters.
    
To discern between the changes, also modify the tag mlflow.set_tag("model", "new-tag-here") where you add
a new tag of your choice. This way you can keep the models apart.

In [75]:
trainer.loop() # if you want to pick up training, loop will continue from the last epoch

100%|██████████| 81/81 [00:03<00:00, 20.64it/s]
2026-04-08 13:51:23.290 | INFO     | mltrainer.trainer:report:181 - Resuming epochs from previous training at 3
2026-04-08 13:51:23.375 | INFO     | mltrainer.trainer:report:215 - Epoch 3 train 0.3613 test 0.2299 metric ['0.9484']
100%|██████████| 81/81 [00:03<00:00, 20.68it/s]
2026-04-08 13:51:27.574 | INFO     | mltrainer.trainer:report:215 - Epoch 4 train 0.1299 test 0.2391 metric ['0.9344']
2026-04-08 13:51:27.576 | INFO     | mltrainer.trainer:__call__:258 - best loss: 0.2299, current loss 0.2391.Counter 1/5.
100%|██████████| 81/81 [00:03<00:00, 20.71it/s]
2026-04-08 13:51:31.766 | INFO     | mltrainer.trainer:report:215 - Epoch 5 train 0.0957 test 0.0969 metric ['0.9688']
100%|██████████| 3/3 [00:12<00:00,  4.22s/it]


In [76]:
mlflow.end_run()

Excercises:

- try to improve the RNN model
- test different things. What works? What does not?
- experiment with either GRU or LSTM layers, create your own models. Have a look at `mltrainer.rnn_models` for inspiration. 
- experiment with adding Conv1D layers. Think about the necessary input-output dimensions of your tensors before and after each layer.

You should be able to get above 90% accuracy with the dataset.
Create a report of 1 a4 about your experiments.

In [77]:
# Ok, first double check amount of classes

In [78]:
all_classes = set()
prev_len = 0

for i, (x, y) in enumerate(train.stream()):
    all_classes.update(y.view(-1).tolist())

    if len(all_classes) == prev_len:
        break
    prev_len = len(all_classes)

    if i >= 100:
        break

print("Number of classes:", len(all_classes))
print("Classes:", sorted(all_classes))

Number of classes: 20
Classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [79]:
# 20 classes, as expected with the code earlier in this script